# Diagnose QCL-SAM accuracy on DGX

Use **QCL DGX (.venv)**. Run cells in order. This notebook performs a fixed-subset overfit experiment; it does not start the 24-epoch stream or alter production training.

Eight train images and eight validation images are sampled with a fixed seed, without augmentation. SAM features are computed once. Three variants share the same initial segmentation head and sample schedule: SAM features + mask priors, quantum features + mask priors, and SAM + quantum residual + mask priors. The quantum branches share initialization. Optimizer, loss and training budget match; trainable parameter counts differ.

Run this after stopping the full training process to avoid GPU contention. Keep the environment and dataset paths from your working training notebook. Results are written to a separate run folder.


In [ ]:
from pathlib import Path
import os
import sys
from datetime import datetime

REPO = Path("/raid/workspace/AI4CV/AI4CV_CL_DGX_A100")
SAM_CHECKPOINT = Path("/raid/workspace/AI4CV/models/sam_vit_b_01ec64.pth")
GPU = "0"
EPOCHS = 24                  # Set to 1 for an initial end-to-end check (not final results).
BATCH_SIZE = 2
NUM_WORKERS = 4              # Set 0 if the server reports shared-memory/worker errors.
SAM_PRECISION = "bfloat16"   # A100 acceleration; "float32" is the baseline.
RESIDUAL_SCALE = 0.1         # SAM features + 0.1 * quantum features; 0 restores legacy behavior.
INSTALL_DEPENDENCIES = False # True only if this environment still needs requirements.txt.

# Leave None to detect from the two dataset locations in your screenshot.
# Set explicit paths here if detection reports missing or ambiguous directories.
OEM_ROOT = None
LANDCOVER_ROOT = None

PROJECT = REPO / "QCL_src"
assert PROJECT.is_dir(), f"Project not found: {PROJECT}"
assert Path(sys.prefix).resolve() == (PROJECT / ".venv").resolve(), (
    f"Select QCL DGX (.venv). Current Python: {sys.executable}, prefix: {sys.prefix}"
)
assert SAM_CHECKPOINT.is_file(), f"Checkpoint not found: {SAM_CHECKPOINT}"
assert EPOCHS >= 1 and BATCH_SIZE >= 1
os.chdir(PROJECT)
os.environ["CUDA_VISIBLE_DEVICES"] = GPU
os.environ["SAM_CHECKPOINT"] = str(SAM_CHECKPOINT)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = PROJECT / "outputs" / ("dgx_notebook_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
RUN_DIR.mkdir(parents=True, exist_ok=False)
VALIDATED = False
TRAINING_COMPLETE = False
print("Python:", sys.executable)
print("Checkpoint:", SAM_CHECKPOINT)
print("Run output:", RUN_DIR)

### Optimized run settings

Frozen SAM uses BF16 on the A100; the quantum circuit and trainable head stay in FP32. Metrics remain on the prediction device. Four data workers prefetch images. SAM is still frozen and is not cached.

The residual path changes the architecture: the head sees SAM's spatial features plus the quantum adaptation, instead of only the coarse quantum output. This is intended to preserve spatial information, not a guarantee of higher accuracy. Set RESIDUAL_SCALE=0 for a baseline comparison. Use each run's saved config when evaluating its checkpoints. Compare per-class IoU and validation metrics, not accuracy alone.


## Environment and GPU

Dependency installation is optional. GPU validation runs in a fresh process, so the selected GPU is respected even if the notebook kernel previously imported PyTorch. All training and validation commands use this kernel's Python and the current checkout.

In [ ]:
import subprocess
import json

if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

# Run CPU regressions and GPU checks (GPU-specific tests skip on CPU-only hosts).
tests = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_diagnostics.py", "-v"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(tests.stdout)
tests.check_returncode()

probe = """
import torch, yaml, pennylane, segment_anything, sklearn, matplotlib
import qcl_sam_seg
print("Package:", qcl_sam_seg.__file__)
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
assert torch.cuda.is_available(), "CUDA unavailable: check the selected GPU and the server PyTorch installation"
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
"""
for command in ([sys.executable, "-u", "-c", probe], ["nvidia-smi"]):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout)
    result.check_returncode()
revision = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
(RUN_DIR / "environment.json").write_text(json.dumps({
    "git_commit": revision, "python": sys.executable, "gpu": GPU,
    "sam_checkpoint": str(SAM_CHECKPOINT), "epochs": EPOCHS, "batch_size": BATCH_SIZE,
}, indent=2))
print("Git commit:", revision)

## Create run-specific configs

Tracked YAML files are preserved. Copies contain absolute DGX paths, the explicit SAM checkpoint, and this run's output directory. Existing split options (including the OpenEarthMap availability filter and holdout) are preserved. If both candidate dataset locations exist, set the desired root in the first cell and rerun setup.

In [ ]:
import copy
import yaml

def choose_root(explicit, relative):
    if explicit is not None:
        path = Path(explicit).expanduser().resolve()
        assert path.is_dir(), f"Dataset directory missing: {path}"
        return path
    candidates = list(dict.fromkeys(
        p.resolve() for p in (REPO / "A100_datasets" / relative,
                              REPO.parent / "A100_datasets" / relative) if p.is_dir()
    ))
    if len(candidates) != 1:
        raise ValueError(f"Set an explicit dataset root for {relative}; found: {candidates}")
    return candidates[0]

roots = {
    "openearthmap": choose_root(OEM_ROOT, Path("openearthmap") / "OpenEarthMap_wo_xBD"),
    "landcoverai": choose_root(LANDCOVER_ROOT, Path("landcover_ai")),
}
CONFIG_DIR = RUN_DIR / "configs"
CONFIG_DIR.mkdir(exist_ok=True)
CONFIGS = {}
for name, root in roots.items():
    source = PROJECT / "configs" / "datasets" / f"{name}.yaml"
    cfg = copy.deepcopy(yaml.safe_load(source.read_text()))
    old_root = (source.parent / cfg["dataset"]["root"]).resolve()
    cfg["dataset"]["root"] = str(root)
    for split, spec in cfg["dataset"]["splits"].items():
        old_value = spec if isinstance(spec, str) else spec["manifest"]
        old_manifest = (source.parent / old_value).resolve()
        try:
            manifest = root / old_manifest.relative_to(old_root)
        except ValueError:
            # A custom manifest outside the dataset retains its original absolute path.
            manifest = old_manifest
        assert manifest.is_file(), f"{name}/{split} manifest not found: {manifest}"
        if isinstance(spec, str):
            cfg["dataset"]["splits"][split] = str(manifest)
        else:
            spec["manifest"] = str(manifest)
    cfg["model"]["sam_checkpoint"] = str(SAM_CHECKPOINT)
    cfg["model"].update(sam_precision=SAM_PRECISION, residual_scale=RESIDUAL_SCALE)
    cfg["training"].update(epochs=EPOCHS, batch_size=BATCH_SIZE,
                           num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=2)
    cfg.setdefault("output", {})["root"] = str(RUN_DIR)
    destination = CONFIG_DIR / f"{name}.yaml"
    destination.write_text(yaml.safe_dump(cfg, sort_keys=False))
    CONFIGS[name] = destination
    print(name, "->", root)

STREAM_NAME = "openearthmap_then_landcoverai"
STREAM = CONFIG_DIR / "stream.yaml"
STREAM.write_text(yaml.safe_dump({
    "name": STREAM_NAME, "tasks": [str(p) for p in CONFIGS.values()]
}, sort_keys=False))
RESULTS = RUN_DIR / STREAM_NAME
print("Stream:", STREAM)

## Validate the datasets

Checks that train/validation/test splits have usable pairs and no shared sample IDs, and scans the configured number of masks for unknown labels. A validation error stops this cell; fix the reported paths or data before training.

In [ ]:
import threading
import queue

def run_logged(arguments, log_path, module="qcl_sam_seg"):
    """Stream child output, save it, and propagate failures or interrupts."""
    command = [sys.executable, "-u", "-m", module, *map(str, arguments)]
    print("Running:", " ".join(command), flush=True)
    messages = queue.Queue()
    with Path(log_path).open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command, cwd=PROJECT, env=os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
        )
        def reader():
            try:
                for line in process.stdout:
                    messages.put(line)
            finally:
                messages.put(None)
        thread = threading.Thread(target=reader, daemon=True)
        thread.start()
        try:
            while True:
                try:
                    line = messages.get(timeout=0.5)
                except queue.Empty:
                    continue
                if line is None:
                    break
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
            if code:
                raise subprocess.CalledProcessError(code, command)
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise
        finally:
            thread.join(timeout=2)
            process.stdout.close()
    print("Saved log:", log_path)

VALIDATED = False
for name, config in CONFIGS.items():
    run_logged(["prepare-data", "--config", config], RUN_DIR / f"validate_{name}.log")
VALIDATED = True
print("Both datasets passed validation.")

## Run controlled diagnostic

Start with OpenEarthMap, where the low scores were observed. STEPS counts optimizer updates, not epochs. Each update uses one fixed cached image, so this is a learning-capacity check, not a throughput benchmark. Increase STEPS if the curves are still decreasing; there is no assumed pass threshold.

Original image/mask dimensions and raw label IDs are checked on selected samples. Inspect the saved overlays yourself: matching dimensions cannot prove semantic alignment. Missing classes and nonrepresentative sampling can affect these small-subset metrics.

In [ ]:
DIAGNOSTIC_DATASET = "openearthmap"  # Change to "landcoverai" for a separate diagnostic.
DIAGNOSTIC_STEPS = 200
DIAGNOSTIC_SAMPLES = 8
DIAGNOSTIC_VAL_SAMPLES = 8
DIAGNOSTIC_SEED = 42
DIAGNOSTIC_LR = 0.001
DIAG_DIR = RUN_DIR / ("diagnostic_" + DIAGNOSTIC_DATASET)
assert VALIDATED
assert not DIAG_DIR.exists(), "Choose a new run directory before repeating."
run_logged([
    "--config", CONFIGS[DIAGNOSTIC_DATASET], "--output", DIAG_DIR,
    "--samples", DIAGNOSTIC_SAMPLES, "--val-samples", DIAGNOSTIC_VAL_SAMPLES,
    "--steps", DIAGNOSTIC_STEPS, "--seed", DIAGNOSTIC_SEED, "--lr", DIAGNOSTIC_LR,
], RUN_DIR / "diagnostic.log", module="qcl_sam_seg.diagnostics")

## Compare results

A SAM-only model that learns the training subset while the quantum-only model does not points toward the quantum path as a limitation. If all variants fail to fit, inspect labels, feature suitability, head capacity, and optimization. These are hypotheses, not automated diagnoses. A gap between training and this small validation subset does not establish full-dataset generalization.

In [ ]:
from IPython.display import display, Markdown, Image
import matplotlib.pyplot as plt
import csv

summary = json.loads((DIAG_DIR / "summary.json").read_text())
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for mode, result in summary.items():
    history = json.loads((DIAG_DIR / f"{mode}_history.json").read_text())
    for axis, split, metric in zip(axes, ("train", "train", "val"), ("loss", "miou", "miou")):
        axis.plot([row["step"] for row in history], [row[split][metric] for row in history], label=mode)
        axis.set(title=f"{split} {metric}", xlabel="Optimizer step")
        axis.grid(alpha=0.2)
        axis.legend()
    print(mode, "| final train mIoU:", round(result["train"]["miou"], 4),
          "| subset val mIoU:", round(result["val"]["miou"], 4))
fig.tight_layout()
fig.savefig(DIAG_DIR / "comparison.png")
display(fig)
plt.close(fig)
for mode in summary:
    display(Markdown(f"### {mode}: per-class results"))
    print((DIAG_DIR / f"{mode}_classes.csv").read_text())
for path in sorted(DIAG_DIR.glob("*_labels.png"))[:4]:
    display(Markdown(path.name))
    display(Image(filename=str(path), width=1000))
for path in sorted(DIAG_DIR.glob("*_prediction.png")):
    display(Markdown(path.name))
    display(Image(filename=str(path), width=1000))

In [ ]:
import zipfile
archive = RUN_DIR / "diagnostic_reports.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in DIAG_DIR.rglob("*"):
        if path.is_file():
            bundle.write(path, path.relative_to(RUN_DIR))
    bundle.write(RUN_DIR / "diagnostic.log", "diagnostic.log")
print("Download with VS Code Remote Explorer:", archive)
print("Share summary.json, comparison.png, class CSVs, and label/prediction overlays for diagnosis.")